# Train DustyLM from Scratch

Train Dusty, a tiny ~8M parameter assistant that talks like a robot vacuum. From raw data to a chatting model.

**What you will do:**
1. Set up the environment (Colab, RunPod, or local)
2. Download the training datasets
3. Train the tokenizer
4. Prepare tokenized data
5. Pretrain the base model
6. Select the best base checkpoint
7. Fine-tune with SFT
8. Select the best SFT checkpoint
9. Chat with your trained model

This notebook works on **Google Colab, RunPod, or any local machine**. The setup cell auto-detects your environment.

> **Note on Execution:** This notebook is designed as an interactive, educational guide.
> It includes manual checkpoint evaluations and is not meant to be run top-to-bottom in a single click.
> If you are looking for a fully automated end-to-end run, open your terminal and execute `make train-end-to-end`.


In [ ]:
import sys, os
from pathlib import Path
import torch

# Ensure uv is available in cloud notebook runtimes such as RunPod.
!pip install -q uv

if "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ:
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        print("Run this from the repo root:\n        uv sync")
REPO_ROOT = Path.cwd()
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Repo root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

---
## 1. Download the Datasets

Downloads TinyStories for pretraining and the Dusty SFT JSONL from [mkhordoo/dusty-chat](https://huggingface.co/datasets/mkhordoo/dusty-chat). If you already have `artifacts/datasets/tinystories_base.txt` and `artifacts/datasets/dusty_sft.jsonl`, skip this cell.

**Why TinyStories?**
The sole purpose of pre-training is to teach basic English grammar and logic. Because our custom tokenizer is intentionally constrained to 4,000 tokens, we need simplified language. The Dusty vacuum persona naturally aligns with the direct, limited vocabulary of a 3-to-4-year-old. TinyStories fits these constraints perfectly, letting the 8M model learn correct English structure without being overwhelmed by complex syntax.

By default this downloads a 50k subset of TinyStories (free Colab friendly).

<details>
<summary>Adjust dataset size</summary>

If you have more system RAM, you can increase the slice size:

`!make download-datasets TINYSTORIES_SLICE="train[:100000]"`

or:

`!make download-datasets TINYSTORIES_SLICE="train[:200000]"`

Free Colab can be tight above 50k-80k, so only use larger slices if your runtime has enough system RAM. To download the full dataset, pass `TINYSTORIES_SLICE="train"`.

</details>

In [ ]:
!make download-datasets

## 2. Check the Training Files

Training needs TinyStories text and SFT data under `artifacts/datasets/`. This cell verifies the required files exist.

In [ ]:
required_sources = {
    "pretrain text": REPO_ROOT / "artifacts" / "datasets" / "tinystories_base.txt",
    "SFT chat JSONL": REPO_ROOT / "artifacts" / "datasets" / "dusty_sft.jsonl",
}

for label, path in required_sources.items():
    status = "✓ found" if path.exists() else "✗ missing"
    print(f"{label:16} {status:10} {path}")

---
## 3. Train the Tokenizer

Before Dusty can learn, it needs to know how to read. Standard tokenizers (like OpenAI's TikToken) use massive vocabularies that would completely overwhelm our tiny model's 8M parameter capacity. Since Dusty only needs to understand simple, child-like English and basic vacuum commands, we will train a custom, highly efficient 4096-token BPE tokenizer from scratch.

This step trains the tokenizer on our local text and saves it to `artifacts/tokenizers/dusty_tokenizer.json`.

> **🚨 Important:** Tokenizer training happens **only** at the very beginning of the pre-training phase. You must **never** retrain the tokenizer before or during the SFT phase. The base model's weights are permanently mapped to specific token IDs; retraining shuffles these IDs, causing the model to output pure garbage. Treat the tokenizer as a strictly fixed artifact for the entire remaining pipeline.


In [ ]:
!make tokenizer
print("✅ Tokenizer successfully trained! Vocabulary is now locked at 4096 tokens.")

## 4. Prepare Tokenized Datasets

Turns raw text and chat JSONL into Hugging Face `datasets` saved on disk. The training loop reads these tokenized datasets directly. ( Takes about ~ 2 mins)

In [ ]:
!make data-pretrain
!make data-sft
print(
    "✅ Data preparation complete! Both pre-training and SFT datasets are successfully tokenized, packed, and ready for the training loop."
)

---
## 5. Pretrain Dusty

Pretraining teaches the base `dusty8m` model language patterns before SFT teaches it to answer user messages. `EPOCHS=1` is a smoke test; increase for a better model.

> **Note:** We stopped pretraining at 5,700 steps (~46M tokens, ~2 epochs) because the model was already generating cohesive English. While Chinchilla scaling laws suggest ~19,600 steps (160M tokens) for an 8M parameter model, this early checkpoint works well for this lightweight demonstration.

**Start TensorBoard first** (non-blocking, runs in background), then start training.


##### 5b. Monitor Training with TensorBoard

The `%tensorboard` magic is non-blocking: it starts the dashboard as a background process and renders the UI immediately. You can run the training cell right after without waiting.

TensorBoard auto-refreshes every 30 seconds. The chart will be empty until training starts producing metrics — click the 🔄 refresh icon once training is running to see the loss curve.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir artifacts/tensorboard


In [ ]:
# Colab T4 can run batch 128 comfortably.
# Reduce BATCH_SIZE=32 if running locally on an older laptop.
EPOCHS = 1
!make train-pretrain EPOCHS={EPOCHS} CHECKPOINT_EVERY_STEPS=100 BATCH_SIZE=128
print(
    f"✅ Pre-training complete! The {EPOCHS}-epoch base model and all step checkpoints have been successfully saved to the artifacts folder."
)

<details>
<summary>💡 Hint: The Chinchilla Scaling Law (Optimum number of EPOCHS)</summary>

DeepMind's research shows that the optimal amount of pre-training data is exactly 20 tokens for every 1 parameter in your model. 

Since Dusty has 8 million parameters, its "perfect" diet is 160 million tokens. With our dataset size, you would need to run this pre-training loop for about 14 epochs to hit that mathematical sweet spot!

</details>

##### 5c. The Full Training Lifecycle Loss Curve

Below is the TensorBoard loss curve from our production runs, showing both phases of training overlaid:

* **Pink Line (Pre-training):** We intentionally implemented early stopping around step 5,700. Monitoring showed the model was already outputting cohesive English, meaning further compute burn wasn't necessary for this baseline.
* **Yellow Line (SFT):** The fine-tuning phase ran for a full ~19,600 steps to strictly enforce the robot vacuum persona and conversational formatting.

If you are running this locally, spin up TensorBoard to ensure your pre-training curve follows a similar downward trajectory before moving to SFT.

<img src="../docs/images/training-lifecycle-loss.png" alt="Dusty pretraining loss curve" width="600"/>


## 6. Check the Pretrained Base Model

Before SFT, generate from the `dusty8m` base profile. This tells you whether the base checkpoint has learned stable text patterns.

In [ ]:
!make generate PROFILE=dusty8m PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 7. Compare Pretraining Step Checkpoints

The training loop saves step checkpoints based on the `CHECKPOINT_EVERY_STEPS=50` value you used during training. If you look inside `artifacts/checkpoints/`, you should see files such as `dusty8m_step_500.pt`.

By default, the final checkpoint is saved as `dusty8m.pt` and used as the base model. However, the last checkpoint is not always the best one. Generate samples from a few step checkpoints, then pick the checkpoint with the best stability and rhythm before
the next step. Replace the step numbers below with checkpoints that exist in your `artifacts/checkpoints/` folder.

**Note:** In production, you would use an evaluation dataset to choose the best checkpoint. We skip that here to keep the workflow simple. For our tiny model, step 5700 produced good results, so we stopped early. You can also let training run for the full
epoch and compare each saved checkpoint.

In [ ]:
for step in [100, 400, 900]:
    print("=" * 80)
    print("CHECKPOINT_STEP:", step)
    !make generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

<details>
<summary>💡 Tip: Automated checkpoint evaluation</summary>

Testing multiple checkpoints by hand gets tedious. If you want to run a bulk evaluation across your training run, see **Automated Checkpoint Evaluation** in the advanced notebook.

</details>


---

## 8. Promote the Best Base Checkpoint

Choose the best checkpoint from Step 7 before we move to the next phase to teach the model its vacuum robot personality.

If a checkpoint performs better than the final one, it needs to be set as the default model by copying it to `artifacts/checkpoints/dusty8m.pt`.

The cell below automates this process for you. All you need to do is set the `BEST_BASE_STEP` variable to your chosen checkpoint number. The code will handle the copying and run a quick generation check to verify it works.

In [ ]:
BEST_PRETRAIN_STEP = 900

!cp artifacts/checkpoints/dusty8m_step_{BEST_PRETRAIN_STEP}.pt artifacts/checkpoints/dusty8m.pt
print(f"✅ Step {BEST_PRETRAIN_STEP} successfully promoted to main base model!")
print("=" * 80)
print("TESTING PROMOTED BASE MODEL:\n")
# Verify the promoted checkpoint
!make generate PROFILE=dusty8m PROMPT="once upon a time" TOP_P=0.9 TEMPERATURE=0.8
print("✅ Generation confirmed. Base model promoted successfully!")

---
## 9. Fine-Tune with SFT

In this step, we teach the model its vacuum robot personality by training it on our synthetic dataset of user-robot conversations. Instead of starting from scratch, this phase builds on the base model you just set as the default. It already understands the English language, but it does not yet know how to carry on a structured chat.

The sft training automatically initializes from `artifacts/checkpoints/dusty8m.pt` base checkpoint (defined in `sft_dusty8m` profile).

> 💡 **Why 21 Epochs?** We are going to run this phase for 21 epochs. Why so many? Because our base model is currently *too smart*. It strongly wants to tell stories about animals and forests. We have to train it on our SFT dataset enough times that it mathematically \"unlearns\" that storytelling bias and fully commits to being a tiny, floor-cleaning robot. Because the SFT dataset is small (only 30k examples), this will still only take a few minutes on a T4 GPU!


In [ ]:
!make train-sft EPOCHS=21 CHECKPOINT_EVERY_STEPS=100

### Training Reference

Here are the numbers from our production runs:

| Metric | Pretrain | SFT |
|---|---|---|
| Dataset examples | 87,586 | 30,000 |
| Batch size | 32 | 32 |
| Max seq len | 256 | 256 |
| Steps per epoch | ~2,738 | ~938 |
| Total steps run | ~5,700 | ~21,500 |
| Best checkpoint | ~5,700 | ~19,600 |
| Tokens seen | ~46M | ~160M |
| Epochs | ~2.1 | ~23 |

If you train with these settings, your results should be similar.


## 10. Compare SFT Step Checkpoints

`EPOCHS=1` is a fast demo: it produces ~938 steps and saves checkpoints at 200, 700, and 900. These checkpoints show early alignment, not final quality. In our full run, the best Dusty persona checkpoint was around step 19,600 (~21 epochs).

##### 10.1 Generation Parameters

Small models have noisy token distributions, so generation settings matter:

| Parameter | Value | Effect |
|---|---|---|
| Temperature | 0.6 - 0.8 | Controls randomness. Lower is repetitive; higher can drift. |
| Top-K | 5 | Keeps only the 5 most likely next tokens. |
| Top-P | 0.8 - 0.9 | Keeps the smallest token set covering 80-90% probability mass. |

The `sft_dusty8m` profile sets these defaults in `dustylm/config.py`; the commands below override `TOP_P` and `TEMPERATURE` explicitly.

In [ ]:
# Sample checkpoints from the 1-epoch local run
for step in [200, 700, 900]:
    print("=" * 80)
    print("SFT CHECKPOINT_STEP:", step)
    !make generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="who are you?" TOP_P=0.9 TEMPERATURE=0.8

<details>
<summary>💡 Tip: Automated checkpoint evaluation</summary>

Testing multiple checkpoints by hand gets tedious. If you want to run a bulk evaluation across your training run, see **Automated Checkpoint Evaluation** in the advanced notebook.

</details>


##### 10.2 What to Look For

Read the 1-epoch outputs as progress signals, not finished behavior:

1. **Step 200:** TinyStories-style narrative text still appears in the responses.
2. **Step 700:** The model starts shifting toward robot/vacuum language.
3. **Step 900:** The chat format and persona are more visible, but still unstable.


> 💡 **Note:** Full alignment takes much longer. We trained 23 epochs total and found the best SFT checkpoint around step 19,600 (~21 epochs). Your exact samples may differ because generation is probabilistic.

## 11. Promote a Checkpoint

The chat cells load the `artifacts/checkpoints/dusty8m_sft.pt` checkpoint. Promotion just copies your chosen step checkpoint to that filename.

Since we ran a short 1-epoch demo, use step 900. The cell below will copy the file and immediately run a quick generation test to verify the model loads correctly.

In [ ]:
BEST_SFT_STEP = 900  # Replace this with your best step

!cp artifacts/checkpoints/dusty8m_sft_step_{BEST_SFT_STEP}.pt artifacts/checkpoints/dusty8m_sft.pt
print(f"✅ Step {BEST_SFT_STEP} successfully promoted to main SFT model!")
# 2. Verify the promotion
print("=" * 80)
print("TESTING PROMOTED SFT MODEL")
!make generate PROFILE=sft_dusty8m PROMPT="who are you?" TOP_P=0.9 TEMPERATURE=0.8
print("✅ Generation confirmed. SFT model promoted successfully!")

---
## 12. Chat with Your Trained Model

Run cell below to chat with your newly trained model


In [ ]:
from dustylm.inference import Inference

engine = Inference(
    checkpoint_path=REPO_ROOT / "artifacts" / "checkpoints" / "dusty8m_sft.pt",
    tokenizer_path=REPO_ROOT / "artifacts" / "tokenizers" / "dusty_tokenizer.json",
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["who are you?", "what makes you happy"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

> **💡 Prefer the CLI?** If you want to chat in the terminal, open your system terminal outside of this notebook, navigate to the repository root, and run:
>
> ```bash
> make chat
> ```



---
## 13. Save Your Checkpoints

Cloud runtimes like Google Colab and RunPod are ephemeral. Your trained checkpoint files will be permanently deleted when the instance shuts down. Download them before closing this notebook.

The cell below downloads your promoted base and sft models. 
  - On **Google Colab** it triggers a browser download for each file. 
  - On **other cloud runtimes** (RunPod, etc.) it prints the file paths and sizes for manual download.



In [ ]:
import sys
from pathlib import Path

checkpoints_dir = Path("artifacts/checkpoints")
promoted = ["dusty8m.pt", "dusty8m_sft.pt"]

if "google.colab" in sys.modules:
    from google.colab import files

    for name in promoted:
        path = checkpoints_dir / name
        if path.exists():
            files.download(str(path))
    print("Downloads triggered. Check your browser downloads folder.")
else:
    for name in promoted:
        path = checkpoints_dir / name
        if path.exists():
            size = path.stat().st_size / 1024 / 1024
            print(f"{path}  {size:.1f} MB")
        else:
            print(f"{path}  not found")
    print(
        "\nDownload these files manually via your cloud provider's file browser or scp."
    )


---

## 14. What's Next?

Congratulations! 🚀 You have successfully trained an LLM from scratch, guided it through a pre-training phase to learn English, and fine-tuned it to adopt a specific persona.

If you want to push this model further, check out the **Advanced Notebook** (`notebooks/03_advanced_tools.ipynb`), where we cover:

* **Changing the Persona:** How to edit the pipeline to make the model act like a different character entirely.

* **Synthetic Data Generation:** How to use API providers to generate your own custom SFT chat examples from scratch.

* **Temperature & Generation Tweaks:** How adjusting sampling parameters can fundamentally change the model's perceived personality.


---

## 15. [Optional] CLI Cheatsheet

<details>
<summary>Click to expand terminal commands</summary>

Now that you understand the mechanics, you don't need a Jupyter notebook to run this pipeline. You can execute the entire flow directly from your terminal using the commands below:

```bash
make download-datasets
make tokenizer
make data-pretrain
make train-pretrain EPOCHS=1
make data-sft
make train-sft EPOCHS=1
make chat
```

>**💡 Smoke Test:** You can also run `make train-end-to-end` to run this entire pipeline sequentially with `EPOCHS=1` hardcoded. It is the perfect way to verify your system runs correctly before scaling up to larger epoch counts!
</details>
